# Test `newsindex` against the real Drive archive

This notebook runs the full `newsindex` pipeline against the **real** scraped-article
archive in Google Drive (`Shak_Scrapped_Top_Articles`, shared at
<https://drive.google.com/drive/u/0/folders/1BF9Y2x9leJJ0lb6ftPmPeYT7Abrk3ygw>), laid
out as `<root>/<country>/<newspaper>/<year-month>/articles_<newspaper>_<year-month>.csv.gz`
-- exactly what `newsindex.read_articles()` expects.

Unlike `test_with_excel.ipynb` (a tiny, hand-computed fixture), this is real data: there
is no "expected" answer to assert against, so this notebook prints and sanity-checks
results instead of asserting exact numbers.

**Scope warning.** The full archive is ~21.5 million articles across 54 countries. Do
not point `DRIVE_ROOT` at the archive root and run this unmodified -- it will take a
very long time and use a lot of memory over a Drive mount. `COUNTRIES` below defaults
to a couple of fast-to-read countries; widen it deliberately once you know the runtime
you're willing to accept.

## 1. Get the archive into this notebook's filesystem

Pick **one** of the two options below depending on where this notebook runs.

**Option A -- Google Colab.** Mounts your Drive at `/content/drive`. Run the cell;
it will prompt you to authorize access once.

**Option B -- local Jupyter, with Google Drive for Desktop synced.** Point
`DRIVE_ROOT` at the synced folder directly -- for this project that has historically
been
`G:\My Drive\News_Articles_Africa_Workbook\Main_Monthly_Scrapper\Scrapped_Top_Articles`
(adjust the drive letter / path to match your own machine).

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Adjust this to wherever "Shak_Scrapped_Top_Articles" lands under your Drive.
    DRIVE_ROOT = Path("/content/drive/MyDrive/Shak_Scrapped_Top_Articles")
else:
    # Local Google Drive for Desktop sync path -- edit this for your machine.
    DRIVE_ROOT = Path(r"G:\My Drive\News_Articles_Africa_Workbook\Main_Monthly_Scrapper\Scrapped_Top_Articles")

print("IN_COLAB:", IN_COLAB)
print("DRIVE_ROOT:", DRIVE_ROOT)
print("exists:", DRIVE_ROOT.exists())

## 2. Install / import `newsindex`

In Colab, install from PyPI. Locally, this notebook assumes it is run from inside a
clone of [CAPE-ERC/newsindex-library](https://github.com/CAPE-ERC/newsindex-library)
(so `data/eptu_country_keywords.json` and `data/gdp_ppp_2015_2025.csv` are reachable by
relative path -- they are not bundled in the PyPI package).

In [ ]:
if IN_COLAB:
    %pip install -q newsindex
    REPO_ROOT = None  # clone the repo in Colab if you also want the bundled data/ files:
    # !git clone https://github.com/CAPE-ERC/newsindex-library.git
    # REPO_ROOT = Path("newsindex-library")
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
    sys.path.insert(0, str(REPO_ROOT / "src"))  # skip this if newsindex is pip-installed

import pandas as pd
import newsindex as ni

print("newsindex", ni.__version__)

## 3. Read a limited slice of the real archive

`newsindex.read_articles()` derives the `country` column from the first path segment
under the directory you point it at, so pointing it at a single country's folder would
mislabel every article with the *newspaper* name instead. We work around that here by
reading each requested country separately and overwriting `country` with the value we
already know.

In [ ]:
# Keep this small on a first run; each entry here is one full country folder.
COUNTRIES = ["Nigeria", "Kenya"]

frames = []
for country in COUNTRIES:
    country_root = DRIVE_ROOT / country
    if not country_root.exists():
        print(f"skipping {country}: {country_root} not found")
        continue
    frame = ni.read_articles(country_root)
    frame["country"] = country  # read_articles() thought the newspaper name was the country
    frames.append(frame)
    print(f"{country}: {len(frame)} articles")

articles = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
articles.head()

## 4. Deduplicate and classify against the production dictionary

In [ ]:
assert len(articles), "no articles were read -- check DRIVE_ROOT and COUNTRIES above"

if REPO_ROOT is not None:
    dict_path = REPO_ROOT / "data" / "eptu_country_keywords.json"
else:
    # Colab without a repo clone: point this at your own copy, e.g. under
    # .../Main_Monthly_Scrapper/EPU_Computations/EPTU_Framework/eptu_country_keywords.json
    dict_path = DRIVE_ROOT.parent / "EPU_Computations" / "EPTU_Framework" / "eptu_country_keywords.json"

kw = ni.dictionary.load(dict_path)
issues = ni.dictionary.validate(kw)
print(f"dictionary covers {len(kw)} countries, {len(issues)} validation issue(s)")
print("fingerprint:", ni.dictionary.fingerprint(kw))

deduped = ni.dedup.dedupe(articles)
print(f"{len(articles) - len(deduped)} duplicate article(s) removed within country-month")

counts = ni.index.count(deduped, kw)
counts.sort_values(["country", "site", "year_month"])

## 5. Fit a baseline and score

Adjust `BASELINE_START` / `BASELINE_END` to match the months actually present in
`counts` above -- the frozen methodology baseline is 2015-01 to 2025-12, but a small
test slice of the archive may not cover that whole window.

In [ ]:
BASELINE_START = str(counts["year_month"].min())
BASELINE_END = str(counts["year_month"].max())
print("fitting baseline over", BASELINE_START, "to", BASELINE_END)

baseline = ni.index.fit_baseline(counts, start=BASELINE_START, end=BASELINE_END)
for country in COUNTRIES:
    retained = baseline.retained_sites(country)
    print(f"{country}: {len(retained)} retained newspaper(s) -> {retained}")

scored = ni.index.score(counts, baseline)
scored.sort_values(["country", "year_month"])

## 6. (Optional) GDP-weighted aggregation

Only meaningful once you have more than one country in `COUNTRIES` above with a shared
baseline window; with a single country the "region" weight is trivially 1.0.

In [ ]:
if REPO_ROOT is not None and len(COUNTRIES) > 1:
    gdp = ni.aggregate.monthly_gdp(REPO_ROOT / "data" / "gdp_ppp_2015_2025.csv", scale=1000)
    group = {"Sample": COUNTRIES}
    regions = ni.aggregate.aggregate(
        scored, gdp, groups=group, baseline=baseline, include_africa=False
    )
    display_cols = regions.sort_values(["group", "year_month"])
    display_cols
else:
    print("skipped: needs REPO_ROOT (for the bundled GDP file) and 2+ countries")

## Done

This ran the exact same `newsindex` functions as `test_with_excel.ipynb` --
`read_articles`, `dedup.dedupe`, `dictionary.load`/`validate`/`fingerprint`,
`index.count`/`fit_baseline`/`score`, and `aggregate.monthly_gdp`/`aggregate` -- but
against real scraped articles pulled from Drive instead of the hand-labeled fixture.
Widen `COUNTRIES` (or point `DRIVE_ROOT` at the full archive) once you're satisfied
with the runtime on a small slice.